# Lab 8B: Quantifiers, Nested Quantifiers, and Database Semantics

## Learning goals
By the end of this lab, students can:
- visualize `∀` as checking every object and `∃` as finding at least one object;
- explain why the order of nested quantifiers matters;
- test quantified De Morgan equivalences over many finite models;
- compare standard first-order semantics with database semantics: unique names, closed world, and domain closure.

## Chapter 8 connection
Chapter 8 introduces universal and existential quantification, shows that `∀x ∃y Loves(x,y)` and `∃y ∀x Loves(x,y)` mean different things, and explains database semantics as a useful alternative when all object names and facts are known.


In [ ]:
# Run this cell first.

from itertools import product
from html import escape
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not installed. Static demonstrations will be shown instead.")


## 1. Quantifiers over a relation matrix

We will treat `Loves(x,y)` as a binary relation over three people. A checked cell means the ordered pair `(x,y)` is in the relation.


In [ ]:
PEOPLE = ["Ann", "Ben", "Cy"]
PAIRS = [(a, b) for a in PEOPLE for b in PEOPLE]


def relation_from_bits(bits, pairs=PAIRS):
    return {pair for i, pair in enumerate(pairs) if (bits >> i) & 1}


def matrix_from_relation(rel):
    return [[1 if (a, b) in rel else 0 for b in PEOPLE] for a in PEOPLE]


def draw_relation_matrix(rel, title="Relation Loves(x,y)"):
    mat = matrix_from_relation(rel)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(mat)
    ax.set_xticks(range(len(PEOPLE)))
    ax.set_xticklabels([f"y={p}" for p in PEOPLE])
    ax.set_yticks(range(len(PEOPLE)))
    ax.set_yticklabels([f"x={p}" for p in PEOPLE])
    ax.set_title(title)
    for i, row in enumerate(mat):
        for j, val in enumerate(row):
            ax.text(j, i, "T" if val else "F", ha="center", va="center", fontsize=14)
    plt.tight_layout()
    plt.show()


def forall(domain, pred):
    return all(pred(x) for x in domain)


def exists(domain, pred):
    return any(pred(x) for x in domain)


def quantifier_report(rel):
    L = lambda a, b: (a, b) in rel
    checks = {
        "∀x ∃y Loves(x,y)  | everyone loves somebody": forall(PEOPLE, lambda x: exists(PEOPLE, lambda y: L(x, y))),
        "∃y ∀x Loves(x,y)  | someone is loved by everyone": exists(PEOPLE, lambda y: forall(PEOPLE, lambda x: L(x, y))),
        "∀x ∀y (Loves(x,y) ⇒ Loves(y,x))  | symmetric": forall(PEOPLE, lambda x: forall(PEOPLE, lambda y: (not L(x, y)) or L(y, x))),
        "∃x ∀y Loves(x,y)  | someone loves everyone": exists(PEOPLE, lambda x: forall(PEOPLE, lambda y: L(x, y))),
    }
    rows = "".join(f"<tr><td><code>{escape(k)}</code></td><td>{v}</td></tr>" for k, v in checks.items())
    display(HTML("<table><tr><th>Formula</th><th>Truth value</th></tr>" + rows + "</table>"))
    return checks

# A counterexample where everyone loves somebody, but no single person is loved by everyone.
cycle_rel = {("Ann", "Ben"), ("Ben", "Cy"), ("Cy", "Ann")}
draw_relation_matrix(cycle_rel, "Counterexample relation")
quantifier_report(cycle_rel)


In [ ]:
def show_quantifier_bits(bits=98):
    rel = relation_from_bits(bits)
    draw_relation_matrix(rel, f"Loves relation encoded by bit pattern {bits}")
    quantifier_report(rel)

if WIDGETS_AVAILABLE:
    slider = widgets.IntSlider(value=98, min=0, max=2 ** len(PAIRS) - 1, step=1, description="bits")
    out = widgets.Output()
    def refresh(change=None):
        with out:
            clear_output(wait=True)
            show_quantifier_bits(slider.value)
    slider.observe(refresh, names="value")
    display(slider, out)
    refresh()
else:
    show_quantifier_bits(98)


## 2. Quantified De Morgan checks

For a unary predicate `P`, Chapter 8 gives the quantified De Morgan equivalences:

- `¬∀x P(x) ≡ ∃x ¬P(x)`
- `¬∃x P(x) ≡ ∀x ¬P(x)`

The code below verifies these equivalences over every possible unary predicate on the three-person domain.


In [ ]:
def unary_predicate_from_bits(bits):
    return {p for i, p in enumerate(PEOPLE) if (bits >> i) & 1}

records = []
for bits in range(2 ** len(PEOPLE)):
    Pset = unary_predicate_from_bits(bits)
    P = lambda x, Pset=Pset: x in Pset
    left1 = not forall(PEOPLE, P)
    right1 = exists(PEOPLE, lambda x, P=P: not P(x))
    left2 = not exists(PEOPLE, P)
    right2 = forall(PEOPLE, lambda x, P=P: not P(x))
    records.append((bits, sorted(Pset), left1 == right1, left2 == right2))

html = "<table><tr><th>bits</th><th>Objects satisfying P</th><th>¬∀P ≡ ∃¬P</th><th>¬∃P ≡ ∀¬P</th></tr>"
for bits, pset, eq1, eq2 in records:
    html += f"<tr><td>{bits}</td><td>{pset}</td><td>{eq1}</td><td>{eq2}</td></tr>"
html += "</table>"
display(HTML(html))
print("All checks passed:", all(eq1 and eq2 for _, _, eq1, eq2 in records))


## 3. Database semantics: 16 possible binary relations over two named objects

Under standard first-order semantics, models may have unnamed objects and many interpretations. Under database semantics:

1. each constant names a distinct object;
2. facts not listed are treated as false;
3. the domain contains only the named objects.

With two constants `R` and `J` and one binary predicate `Rel`, there are four possible ordered pairs and therefore `2^4 = 16` possible relation extensions.


In [ ]:
OBJECTS_2 = ["R", "J"]
PAIRS_2 = [(a, b) for a in OBJECTS_2 for b in OBJECTS_2]


def rel2_from_bits(bits):
    return {pair for i, pair in enumerate(PAIRS_2) if (bits >> i) & 1}


def draw_db_model(bits, ax=None, title=None):
    rel = rel2_from_bits(bits)
    own_ax = ax is None
    if own_ax:
        fig, ax = plt.subplots(figsize=(3, 3))
    pos = {"R": (0.25, 0.5), "J": (0.75, 0.5)}
    ax.set_title(title if title is not None else f"bits={bits}", fontsize=9)
    ax.axis("off")
    for obj, (x, y) in pos.items():
        ax.add_patch(plt.Circle((x, y), 0.09, fill=False, lw=2))
        ax.text(x, y, obj, ha="center", va="center", fontsize=10)
    for a, b in rel:
        if a == b:
            x, y = pos[a]
            ax.annotate("", xy=(x + 0.04, y + 0.08), xytext=(x - 0.04, y + 0.08),
                        arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=1.6", lw=1.4))
        else:
            ax.annotate("", xy=pos[b], xytext=pos[a], arrowprops=dict(arrowstyle="->", lw=1.5))
    if own_ax:
        plt.show()

fig, axes = plt.subplots(4, 4, figsize=(8, 7))
for bits, ax in enumerate(axes.ravel()):
    draw_db_model(bits, ax=ax, title=f"{bits:04b}")
plt.suptitle("All 16 database-semantics models for one binary relation over {R,J}")
plt.tight_layout()
plt.show()


In [ ]:
def db_semantics_demo(bits=3):
    rel = rel2_from_bits(bits)
    draw_db_model(bits, title=f"Database model {bits:04b}")
    html = "<h3>Facts true under closed-world database semantics</h3><ul>"
    for pair in PAIRS_2:
        html += f"<li><code>Rel{pair}</code> = {pair in rel}</li>"
    html += "</ul>"
    html += "<p>Under closed-world semantics, all missing atomic facts are false. Under standard FOL semantics, missing facts are simply not asserted unless the KB rules them out.</p>"
    display(HTML(html))

if WIDGETS_AVAILABLE:
    s = widgets.IntSlider(value=3, min=0, max=15, step=1, description="model")
    out = widgets.Output()
    def update(change=None):
        with out:
            clear_output(wait=True)
            db_semantics_demo(s.value)
    s.observe(update, names="value")
    display(s, out)
    update()
else:
    db_semantics_demo(3)


## Student practice

1. Find a relation where `∀x ∃y Loves(x,y)` is true and `∃y ∀x Loves(x,y)` is false.
2. Find a relation where both formulas are true.
3. In the database-semantics section, choose a bit pattern and manually list every true ordered pair.
4. Explain why database semantics is convenient for complete databases but risky for incomplete world knowledge.
